# Explore the similarity between subjects using EEGNET, DNN and CCA-based methods

## Loading the data

We first load the data for all subjects and sessions, and prepare it for analysis.

The preprocessing steps include:
- Bandpass filtering the EEG signals to focus on the frequencies of interest located between 8 and 15.8 Hz and up to the third harmonic: 6-50Hz
- CAR (Common Average Reference) spatial filtering
- Normalization of the data to have zero mean and unit variance


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("/home/mateuschinelatto/Experiments/ssvep-bci-nn/cross-subject")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmark_dataset import (
    load_data_from_users,
    load_freq_phase
)

DATASET_ROOT_PATH = Path("/home/mateuschinelatto/Experiments/data/benchmark/")

In [ ]:
# Configuration parameters
users = list(range(1, 36))
users_to_run = users.copy()

# Electrode indices for the electrodes Pz, PO5, PO3, POz, PO4, PO6, O1, Oz, O2
all_occipital_electrodes = np.array([47, 53, 54, 55, 56, 57, 60, 61, 62])
# occipital_electrodes = [61]
occipital_electrodes = np.array([47, 53, 54, 55, 56, 57, 60, 61, 62])
# Which frequencies to analyze
all_frequencies, phases = load_freq_phase(DATASET_ROOT_PATH.joinpath("Freq_Phase.mat"))
print("All frequencies in the dataset:", all_frequencies)
bci_frequencies = all_frequencies[:8]
indices = [np.where(all_frequencies == freq)[0][0] for freq in bci_frequencies]

# Dataset visual delay and sampling frequency
visual_delay = 160 # ms
sample_rate = 250 # Hz

# Bandpass filter configuration
freq_cut_low = 6
freq_cut_high = 50
filter_order = 10

# Optional CAR configuration on loaded data
apply_car = True
car_reference_channels = all_occipital_electrodes
car_target_channels = occipital_electrodes

print("Users of interest:", users)
print("Users to run:", users_to_run)
print("BCI frequencies:", bci_frequencies)
print("Indices of frequencies of interest:", indices)

In [ ]:
# all_data_CAR_norm = load_data_from_users(
#     dataset_path="/home/mateuschinelatto/Experiments/data/benchmark/",
#     users=users,
#     visual_delay=visual_delay,
#     filter_bandpass=True,
#     apply_car=True,
#     car_reference_channels=car_reference_channels,
#     car_target_channels=car_target_channels,
#     sample_rate=sample_rate,
#     freq_cut_low=freq_cut_low,
#     freq_cut_high=freq_cut_high,
#     filter_order=filter_order,
#     normalize=True
# )

# all_data = load_data_from_users(
#     dataset_path="/home/mateuschinelatto/Experiments/data/benchmark/",
#     users=users,
#     visual_delay=visual_delay,
#     filter_bandpass=True,
#     apply_car=False,
#     sample_rate=sample_rate,
#     freq_cut_low=freq_cut_low,
#     freq_cut_high=freq_cut_high,
#     filter_order=filter_order,
# )

Shapes:
- each list position is related to respective user so: [subj_1,subj_2,subj_3,...,subj_n]
- each user has the shape (64, 1250, 40, 6) <-> (channels, time_points, frequencies, trials).
- 1250 time points -> 5s at 250 Hz

We can also get windows of 1s for example:
- each list position is related to respective user so: [subj_1,subj_2,subj_3,...,subj_n]
- each user has the shape (64, 250, 40, 30) <-> (channels, time_points, frequencies, trials*windows).
- 1250 time points -> 5s at 250 Hz -> 5 1s windows

We can also get windows of 1s but without windowing, just getting the first 250 time points of each trial:
- each list position is related to respective user so: [subj_1,subj_2,subj_3,...,subj_n]
- each user has the shape (64, 250, 40, 6) <-> (channels, time_points, frequencies, trials).
- 250 time points -> 1s at 250 Hz -> 1 1s window

In [ ]:
window = 1.0

all_data_CAR_norm_1s = load_data_from_users(
    dataset_path="/home/mateuschinelatto/Experiments/data/benchmark/",
    users=users,
    visual_delay=visual_delay,
    filter_bandpass=True,
    apply_car=True,
    car_reference_channels=car_reference_channels,
    car_target_channels=car_target_channels,
    sample_rate=sample_rate,
    freq_cut_low=freq_cut_low,
    freq_cut_high=freq_cut_high,
    filter_order=filter_order,
    normalize=True,
    window_mode="single",
    window_size=sample_rate*window,
    window_overlap=0
)
print(all_data_CAR_norm_1s[0].shape)

## TSNE Visualization

TSNE is a stochastic method that projects multidimensional data into two dimensions maintaining distance proportions between original and projected samples

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns
from ssvep_shared import build_tensors_no_cca, build_tensors_with_cca

tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)

In [ ]:
# Transform all data into an array to fit TSNE

train_data = np.concatenate(all_data_CAR_norm_1s, axis=-1)
test_data = np.empty(train_data.shape)

x_train, _, labels_freq, _, channels_for_model = (
    build_tensors_no_cca(
        train_data,
        test_data,
        occipital_electrodes,
        bci_frequencies,
        indices,
        250,
        apply_subband_filter=False,
    )
)

x_train_cca, _, labels_freq_cca, _, channels_for_model_cca = (
    build_tensors_with_cca(
        train_data,
        test_data,
        occipital_electrodes,
        bci_frequencies,
        phases,
        indices,
        3,
        0,
        250,
        apply_subband_filter=False,
    )
)

labels_user = []
for user_idx, user_data in enumerate(all_data_CAR_norm_1s):
    num_trials = user_data.shape[3]
    for sessao in range(num_trials):
        for freq_idx in range(len(indices)):
            labels_user.append(users[user_idx])

# Build user labels for CCA order (frequency -> trial), matching x_train_cca from build_tensors_with_cca
labels_user_by_trial = []
for user_idx, user_data in enumerate(all_data_CAR_norm_1s):
    num_trials = user_data.shape[3]
    labels_user_by_trial.extend([users[user_idx]] * num_trials)

labels_user_cca = []
for _ in range(len(indices)):
    labels_user_cca.extend(labels_user_by_trial)


In [ ]:
print(train_data.shape)
print(x_train.shape)
print(x_train_cca.shape)
print(len(labels_freq))
print(len(labels_user))

In [ ]:
X_tsne = tsne.fit_transform(x_train.reshape(x_train.shape[0], -1))
X_tsne_cca = tsne.fit_transform(x_train_cca.reshape(x_train_cca.shape[0], -1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), dpi=300)
sns.set_theme(style="white", font_scale=1.0)

sns.scatterplot(
    x=X_tsne[:, 0],
    y=X_tsne[:, 1],
    hue=labels_freq,
    palette="tab10",
    s=50,
    edgecolor="w",
    alpha=0.85,
    ax=axes[0],
)
axes[0].set_title("t-SNE by Frequency (Without CCA)")
axes[0].legend(title="Stimulus (Hz)", loc="upper left")

sns.scatterplot(
    x=X_tsne_cca[:, 0],
    y=X_tsne_cca[:, 1],
    hue=labels_freq_cca,
    palette="tab10",
    s=50,
    edgecolor="w",
    alpha=0.85,
    ax=axes[1],
)
axes[1].set_title("t-SNE by Frequency (With CCA)")
axes[1].legend(title="Stimulus (Hz)", loc="upper left")

for ax in axes:
    ax.set_xlabel("t-SNE Component 1")
    ax.set_ylabel("t-SNE Component 2")

plt.tight_layout()
plt.show()

In [ ]:
# Plot users in two graphs: without CCA and with CCA
import pandas as pd

# labels_user is ordered as (trial -> frequency), matching x_train from build_tensors_no_cca
df_tsne = pd.DataFrame({
    "tsne_0": X_tsne[:, 0],
    "tsne_1": X_tsne[:, 1],
    "Frequency": labels_freq,
    "User": labels_user,
})

df_tsne_cca = pd.DataFrame({
    "tsne_0": X_tsne_cca[:, 0],
    "tsne_1": X_tsne_cca[:, 1],
    "Frequency": labels_freq_cca,
    "User": labels_user_cca,
})

# Build a deterministic, unique color per user
unique_users = sorted(df_tsne["User"].unique())
user_palette = dict(zip(unique_users, sns.color_palette("husl", n_colors=len(unique_users))))

fig, axes = plt.subplots(1, 2, figsize=(20, 8), dpi=180)
sns.set_theme(style="whitegrid", font_scale=1.05)

sns.scatterplot(
    data=df_tsne,
    x="tsne_0",
    y="tsne_1",
    hue="User",
    palette=user_palette,
    s=45,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.8,
    ax=axes[0],
)
axes[0].set_title("t-SNE by User (Without CCA)")
axes[0].set_xlabel("t-SNE Component 1")
axes[0].set_ylabel("t-SNE Component 2")
axes[0].legend(title="User", bbox_to_anchor=(1.02, 1), loc="upper left")

sns.scatterplot(
    data=df_tsne_cca,
    x="tsne_0",
    y="tsne_1",
    hue="User",
    palette=user_palette,
    s=45,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.8,
    ax=axes[1],
)
axes[1].set_title("t-SNE by User (With CCA)")
axes[1].set_xlabel("t-SNE Component 1")
axes[1].set_ylabel("t-SNE Component 2")
axes[1].legend(title="User", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

## UMAP Visualization

In [ ]:
import umap.umap_ as umap
import pandas as pd


def make_umap_model():
    return umap.UMAP(
        n_components=2,
        n_neighbors=30,
        min_dist=0.1,
        metric="euclidean",
        random_state=42,
    )


X_umap = make_umap_model().fit_transform(x_train.reshape(x_train.shape[0], -1))
X_umap_cca = make_umap_model().fit_transform(
    x_train_cca.reshape(x_train_cca.shape[0], -1)
)

# Frequency visualization (same structure used in t-SNE section)
fig, axes = plt.subplots(1, 2, figsize=(18, 7), dpi=300)
sns.set_theme(style="white", font_scale=1.0)

sns.scatterplot(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    hue=labels_freq,
    palette="tab10",
    s=50,
    edgecolor="w",
    alpha=0.85,
    ax=axes[0],
)
axes[0].set_title("UMAP by Frequency (Without CCA)")
axes[0].legend(title="Stimulus (Hz)", loc="upper left")

sns.scatterplot(
    x=X_umap_cca[:, 0],
    y=X_umap_cca[:, 1],
    hue=labels_freq_cca,
    palette="tab10",
    s=50,
    edgecolor="w",
    alpha=0.85,
    ax=axes[1],
)
axes[1].set_title("UMAP by Frequency (With CCA)")
axes[1].legend(title="Stimulus (Hz)", loc="upper left")

for ax in axes:
    ax.set_xlabel("UMAP Component 1")
    ax.set_ylabel("UMAP Component 2")

plt.tight_layout()
plt.show()

# User visualization (same structure used in t-SNE section)
df_umap = pd.DataFrame(
    {
        "umap_0": X_umap[:, 0],
        "umap_1": X_umap[:, 1],
        "Frequency": labels_freq,
        "User": labels_user,
    }
)

df_umap_cca = pd.DataFrame(
    {
        "umap_0": X_umap_cca[:, 0],
        "umap_1": X_umap_cca[:, 1],
        "Frequency": labels_freq_cca,
        "User": labels_user_cca,
    }
)

unique_users = sorted(df_umap["User"].unique())
user_palette = dict(
    zip(unique_users, sns.color_palette("husl", n_colors=len(unique_users)))
)

fig, axes = plt.subplots(1, 2, figsize=(20, 8), dpi=180)
sns.set_theme(style="whitegrid", font_scale=1.05)

sns.scatterplot(
    data=df_umap,
    x="umap_0",
    y="umap_1",
    hue="User",
    palette=user_palette,
    s=45,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.8,
    ax=axes[0],
)
axes[0].set_title("UMAP by User (Without CCA)")
axes[0].set_xlabel("UMAP Component 1")
axes[0].set_ylabel("UMAP Component 2")
axes[0].legend(title="User", bbox_to_anchor=(1.02, 1), loc="upper left")

sns.scatterplot(
    data=df_umap_cca,
    x="umap_0",
    y="umap_1",
    hue="User",
    palette=user_palette,
    s=45,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.8,
    ax=axes[1],
)
axes[1].set_title("UMAP by User (With CCA)")
axes[1].set_xlabel("UMAP Component 1")
axes[1].set_ylabel("UMAP Component 2")
axes[1].legend(title="User", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

## PCA Visualization

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd

# PCA directly on flattened time-domain samples
pca = PCA(n_components=2)
pca_cca = PCA(n_components=2)

X_pca = pca.fit_transform(x_train.reshape(x_train.shape[0], -1))
X_pca_cca = pca_cca.fit_transform(x_train_cca.reshape(x_train_cca.shape[0], -1))

df_pca = pd.DataFrame(
    {
        "pca_0": X_pca[:, 0],
        "pca_1": X_pca[:, 1],
        "Frequency": labels_freq,
        "User": labels_user,
    }
)

df_pca_cca = pd.DataFrame(
    {
        "pca_0": X_pca_cca[:, 0],
        "pca_1": X_pca_cca[:, 1],
        "Frequency": labels_freq_cca,
        "User": labels_user_cca,
    }
)

unique_users = sorted(df_pca["User"].unique())
user_palette = dict(zip(unique_users, sns.color_palette("husl", n_colors=len(unique_users))))

# User-based visualization
fig, axes = plt.subplots(1, 2, figsize=(20, 8), dpi=180)
sns.set_theme(style="whitegrid", font_scale=1.05)

sns.scatterplot(
    data=df_pca,
    x="pca_0",
    y="pca_1",
    hue="User",
    palette=user_palette,
    s=45,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.8,
    ax=axes[0],
)
axes[0].set_title(
    f"PCA by User (Without CCA)\nExplained variance: {pca.explained_variance_ratio_[:2].sum():.2%}"
)
axes[0].set_xlabel("PCA Component 1")
axes[0].set_ylabel("PCA Component 2")
axes[0].legend(title="User", bbox_to_anchor=(1.02, 1), loc="upper left")

sns.scatterplot(
    data=df_pca_cca,
    x="pca_0",
    y="pca_1",
    hue="User",
    palette=user_palette,
    s=45,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.8,
    ax=axes[1],
)
axes[1].set_title(
    f"PCA by User (With CCA)\nExplained variance: {pca_cca.explained_variance_ratio_[:2].sum():.2%}"
)
axes[1].set_xlabel("PCA Component 1")
axes[1].set_ylabel("PCA Component 2")
axes[1].legend(title="User", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

# Frequency-based visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7), dpi=300)
sns.set_theme(style="white", font_scale=1.0)

sns.scatterplot(
    data=df_pca,
    x="pca_0",
    y="pca_1",
    hue="Frequency",
    palette="tab10",
    s=50,
    edgecolor="w",
    alpha=0.85,
    ax=axes[0],
)
axes[0].set_title("PCA by Frequency (Without CCA)")
axes[0].legend(title="Stimulus (Hz)", loc="upper left")

sns.scatterplot(
    data=df_pca_cca,
    x="pca_0",
    y="pca_1",
    hue="Frequency",
    palette="tab10",
    s=50,
    edgecolor="w",
    alpha=0.85,
    ax=axes[1],
)
axes[1].set_title("PCA by Frequency (With CCA)")
axes[1].legend(title="Stimulus (Hz)", loc="upper left")

for ax in axes:
    ax.set_xlabel("PCA Component 1")
    ax.set_ylabel("PCA Component 2")

plt.tight_layout()
plt.show()

## Frequency analysis + PCA

In [ ]:
# FFT-based features + PCA (focused on 6-50 Hz band)
def fft_pca_features(samples, fs, fmin=6, fmax=50):
    freqs = np.fft.rfftfreq(samples.shape[-1], d=1 / fs)
    band_mask = (freqs >= fmin) & (freqs <= fmax)
    fft_vals = np.fft.rfft(samples, axis=-1)
    amplitudes = np.abs(fft_vals[..., band_mask])
    return amplitudes.reshape(samples.shape[0], -1)

fft_features = fft_pca_features(x_train, sample_rate)
fft_features_cca = fft_pca_features(x_train_cca, sample_rate)

pca_fft = PCA(n_components=2)
pca_fft_cca = PCA(n_components=2)

X_pca_fft = pca_fft.fit_transform(fft_features)
X_pca_fft_cca = pca_fft_cca.fit_transform(fft_features_cca)

df_pca_fft = pd.DataFrame(
    {
        "pca_0": X_pca_fft[:, 0],
        "pca_1": X_pca_fft[:, 1],
        "Frequency": labels_freq,
        "User": labels_user,
    }
)

df_pca_fft_cca = pd.DataFrame(
    {
        "pca_0": X_pca_fft_cca[:, 0],
        "pca_1": X_pca_fft_cca[:, 1],
        "Frequency": labels_freq_cca,
        "User": labels_user_cca,
    }
)

unique_users_fft = sorted(df_pca_fft["User"].unique())
user_palette_fft = dict(zip(unique_users_fft, sns.color_palette("husl", n_colors=len(unique_users_fft))))

# User-based visualization
fig, axes = plt.subplots(1, 2, figsize=(20, 8), dpi=180)
sns.set_theme(style="whitegrid", font_scale=1.05)

sns.scatterplot(
    data=df_pca_fft,
    x="pca_0",
    y="pca_1",
    hue="User",
    palette=user_palette_fft,
    s=45,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.8,
    ax=axes[0],
)
axes[0].set_title(
    f"PCA on FFT Features by User (Without CCA)\nExplained variance: {pca_fft.explained_variance_ratio_[:2].sum():.2%}"
)
axes[0].set_xlabel("PCA Component 1")
axes[0].set_ylabel("PCA Component 2")
axes[0].legend(title="User", bbox_to_anchor=(1.02, 1), loc="upper left")

sns.scatterplot(
    data=df_pca_fft_cca,
    x="pca_0",
    y="pca_1",
    hue="User",
    palette=user_palette_fft,
    s=45,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.8,
    ax=axes[1],
)
axes[1].set_title(
    f"PCA on FFT Features by User (With CCA)\nExplained variance: {pca_fft_cca.explained_variance_ratio_[:2].sum():.2%}"
)
axes[1].set_xlabel("PCA Component 1")
axes[1].set_ylabel("PCA Component 2")
axes[1].legend(title="User", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

# Frequency-based visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7), dpi=300)
sns.set_theme(style="white", font_scale=1.0)

sns.scatterplot(
    data=df_pca_fft,
    x="pca_0",
    y="pca_1",
    hue="Frequency",
    palette="tab10",
    s=50,
    edgecolor="w",
    alpha=0.85,
    ax=axes[0],
)
axes[0].set_title("PCA on FFT Features by Frequency (Without CCA)")
axes[0].legend(title="Stimulus (Hz)", loc="upper left")

sns.scatterplot(
    data=df_pca_fft_cca,
    x="pca_0",
    y="pca_1",
    hue="Frequency",
    palette="tab10",
    s=50,
    edgecolor="w",
    alpha=0.85,
    ax=axes[1],
)
axes[1].set_title("PCA on FFT Features by Frequency (With CCA)")
axes[1].legend(title="Stimulus (Hz)", loc="upper left")

for ax in axes:
    ax.set_xlabel("PCA Component 1")
    ax.set_ylabel("PCA Component 2")

plt.tight_layout()
plt.show()